#  Products - Bronze Ingestion

## Imports


In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, IntegerType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_products"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist"
source_dataset = "products"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,null,/Volumes/ecommerce_dev/landing/raw_files/olist/products/olist_products_dataset.csv,2026-08-02T21:30:33.000Z,2026-08-02T23:02:20.788Z,2f6d716c-6ff2-4d76-9198-d6c7fd8291a0,olist,products


In [0]:
spark.table(target_table).count()

32951